# Week 7: From Baselines to Targeted Experiments

## Objective
In this notebook, we train tree-based models on the cleaned UHPC dataset using a train/validation/test split, then run targeted experiments.

## Goals
- Separate features and target
- Build train/validation/test split
- Train Random Forest and Extra Trees
- Compare results using R2 and RMSE
- Run targeted experiments and justify the baseline

# Step 1: Import Libraries

In this step, we import all the libraries needed for loading data, splitting, training models, and scoring.

In [115]:
# Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.tree import DecisionTreeRegressor

print("Libraries loaded successfully")

Libraries loaded successfully


# Step 2: Load the Cleaned Dataset

In this step, we load the cleaned UHPC dataset created in Week 5. This data is already cleaned, so we do not clean it again.

In [116]:
# Load the cleaned UHPC dataset from Week 5

df = pd.read_csv("semantic_recoding_features_50.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (2072, 37)


,Unnamed: 0,cement,cement_type,cement_grade,silica_fume,fly_ash,fly_ash_type,limestone_powder,quartz_powder,glass_powder,...,fiber1_diameter,fiber2_type,fiber2_amount,water,sp_type,sp_amount,curing_method,curing_temp,cs_28d,cement_type_clean
0,0,839.0,OPC_unknown,NaN,104.0,52.0,class F,0.0,0.0,0.0,...,0.2,not_applicable,0.0,147.0,PCE_SP,59.33,Standard Curing,NaN,132.0,OPC_I
1,1,839.0,OPC_unknown,NaN,104.0,26.0,class F,0.0,0.0,0.0,...,0.2,not_applicable,0.0,147.0,PCE_SP,59.33,Standard Curing,NaN,122.5,OPC_I
2,2,839.0,OPC_unknown,NaN,104.0,0.0,not_applicable,0.0,0.0,0.0,...,0.2,not_applicable,0.0,147.0,PCE_SP,62.15,Standard Curing,NaN,116.0,OPC_I
3,3,839.0,OPC_unknown,NaN,104.0,52.0,class F,0.0,0.0,0.0,...,0.2,not_applicable,0.0,147.0,PCE_SP,64.98,Standard Curing,NaN,134.0,OPC_I
4,4,839.0,OPC_unknown,NaN,104.0,26.0,class F,0.0,0.0,0.0,...,0.2,not_applicable,0.0,147.0,PCE_SP,64.98,Standard Curing,NaN,131.5,OPC_I


# Step 3: Separate Features and Target

In this step, we split the data into features (X) and target (y).

- Target (y) = 28-Day compressive strength
- Features (X) = mix ingredients
- We remove junk columns: row numbers, empty columns, and other results

We exclude arbitrary identifiers and other strength outcomes, keeping only the mix ingredients as predictors.

In [117]:
# Junk columns we remove (not useful for prediction)
drop_columns = ["Unnamed: 0"]

# Target column we want to predict
target_column = "cs_28d"

# Features (X) and target (y)
X = df.drop(columns=drop_columns + [target_column])
y = df[target_column]

# Drop rows where target is missing
missing_target = y.isnull()
X = X[~missing_target]
y = y[~missing_target]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Missing in y:", y.isnull().sum())

Features shape: (2072, 35)
Target shape: (2072,)
Missing in y: 0


# Step 4: Check Missing Values

- Check how many missing values are in each feature
- Models cannot train on missing values
- We identify which columns need to be filled

In [118]:
# Check missing values in features

missing = X.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("Columns with missing values:")
print(missing)

Columns with missing values:
cement_grade       798
fiber1_length      676
fiber1_diameter    675
sand_max_size      276
curing_temp        157
cement_type         19
sp_amount            8
dtype: int64


# Step 5: Handle Missing Values

We fill missing values after splitting to avoid data leakage.

- Number columns → fill with 0 (ingredient was not used)
- Text columns → fill with "None" (not applicable)
- We only use train data to decide fill values

In [119]:
# Identify column types BEFORE any changes
number_cols = X.select_dtypes(include="number").columns.tolist()
text_cols = X.select_dtypes(include=["object", "str"]).columns.tolist()

print("Numeric columns:", len(number_cols))
print("Text columns:", len(text_cols))
print("Text columns list:", text_cols)

Numeric columns: 25
Text columns: 10
Text columns list: ['cement_type', 'fly_ash_type', 'slag_type', 'filler_type', 'sand_type', 'fiber1_type', 'fiber2_type', 'sp_type', 'curing_method', 'cement_type_clean']


# Step 6: Check Text Column Labels

- Count unique labels in each text column
- Columns with too many labels will be dropped
- High-cardinality columns create noise and sparse data

In [120]:
# Check how many unique labels each text column has

for col in text_cols:
    print(col, "→", X[col].nunique(), "labels")

cement_type → 11 labels
fly_ash_type → 4 labels
slag_type → 6 labels
filler_type → 19 labels
sand_type → 23 labels
fiber1_type → 12 labels
fiber2_type → 9 labels
sp_type → 8 labels
curing_method → 7 labels
cement_type_clean → 12 labels


# Step 7: Encode Text Columns

- Text columns have manageable label counts (4 to 23)
- We use one-hot encoding to convert them to numbers
- We drop cement_type_clean as it duplicates cement_type

In [121]:
# Remove cement_type_clean from dataset and text_cols list
if "cement_type_clean" in X.columns:
    X = X.drop(columns=["cement_type_clean"])

# Update text_cols to remove cement_type_clean
text_cols = [c for c in text_cols if c != "cement_type_clean"]

print("Text columns after fix:", text_cols)

Text columns after fix: ['cement_type', 'fly_ash_type', 'slag_type', 'filler_type', 'sand_type', 'fiber1_type', 'fiber2_type', 'sp_type', 'curing_method']


# Step 8: Train / Validation / Test Split

- Split data into 3 parts: train (60%), validation (20%), test (20%)
- Train: model learns from this
- Validation: used to tune and compare models
- Test: final honest score, used only once

In [122]:
# Split first
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42)

# Fill missing values on train only
X_train[number_cols] = X_train[number_cols].fillna(0)
X_val[number_cols]   = X_val[number_cols].fillna(0)
X_test[number_cols]  = X_test[number_cols].fillna(0)

X_train[text_cols] = X_train[text_cols].fillna("None")
X_val[text_cols]   = X_val[text_cols].fillna("None")
X_test[text_cols]  = X_test[text_cols].fillna("None")

# Encode on train only, apply same to val and test
X_train = pd.get_dummies(X_train, columns=list(text_cols), drop_first=True)
train_columns = X_train.columns.tolist()
X_val  = pd.get_dummies(X_val,  columns=list(text_cols), drop_first=True).reindex(columns=train_columns, fill_value=0)
X_test = pd.get_dummies(X_test, columns=list(text_cols), drop_first=True).reindex(columns=train_columns, fill_value=0)

print("Train size:      ", X_train.shape[0])
print("Validation size: ", X_val.shape[0])
print("Test size:       ", X_test.shape[0])
print("Missing in train:", X_train.isnull().sum().sum())

Train size:       1242
Validation size:  415
Test size:        415
Missing in train: 0


# Step 9: Decision Tree (Baseline)

- Decision Tree is our simplest model and the baseline
- We train on training data and check score on validation
- Expected to overfit — single tree memorizes training data

In [123]:
# Train the Decision Tree (our baseline model)

dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)

# Predict on validation data
dt_val_pred = dt_model.predict(X_val)

# Check the scores
dt_r2 = r2_score(y_val, dt_val_pred)
dt_rmse = np.sqrt(mean_squared_error(y_val, dt_val_pred))

print("Decision Tree (Baseline)")
print("Validation R2:  ", round(dt_r2, 4))
print("Validation RMSE:", round(dt_rmse, 4))

Decision Tree (Baseline)
Validation R2:   0.7208
Validation RMSE: 19.9421


# Step 10: Random Forest Model

- Random Forest builds 100 decision trees and averages predictions
- Reduces overfitting compared to single Decision Tree
- Expected to outperform the baseline significantly

In [124]:
# Train Random Forest model

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict on validation data
rf_val_pred = rf_model.predict(X_val)

# Check the scores
rf_r2 = r2_score(y_val, rf_val_pred)
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_val_pred))

print("Random Forest")
print("Validation R2:  ", round(rf_r2, 4))
print("Validation RMSE:", round(rf_rmse, 4))

Random Forest
Validation R2:   0.8398
Validation RMSE: 15.1079


# Step 11: Extra Trees Model

- Extra Trees uses random split thresholds unlike Random Forest
- Extra randomness reduces overfitting further
- We compare against Decision Tree and Random Forest

In [125]:
# Train Extra Trees model

et_model = ExtraTreesRegressor(n_estimators=100, random_state=42)
et_model.fit(X_train, y_train)

# Predict on validation data
et_val_pred = et_model.predict(X_val)

# Check the scores
et_r2 = r2_score(y_val, et_val_pred)
et_rmse = np.sqrt(mean_squared_error(y_val, et_val_pred))

print("Extra Trees")
print("Validation R2:  ", round(et_r2, 4))
print("Validation RMSE:", round(et_rmse, 4))

Extra Trees
Validation R2:   0.8624
Validation RMSE: 14.0006


# Step 12: Model Comparison

- We compare all three models on validation data
- Decision Tree is the baseline
- Random Forest and Extra Trees are ensemble improvements
- Best model will be evaluated on the test set

In [126]:
# Compare all three models

results = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest", "Extra Trees"],
    "Validation R2": [dt_r2, rf_r2, et_r2],
    "Validation RMSE": [dt_rmse, rf_rmse, et_rmse]
})

print(results.to_string(index=False))

        Model  Validation R2  Validation RMSE
Decision Tree       0.720810        19.942123
Random Forest       0.839763        15.107852
  Extra Trees       0.862391        14.000553


# Step 13: Final Evaluation on Test Set

- Extra Trees is our best model (R2: 0.86)
- We evaluate it on the test set for the final honest score
- Test set was never seen by the model before

In [127]:
# Evaluate best model (Extra Trees) on test set

et_test_pred = et_model.predict(X_test)

et_test_r2 = r2_score(y_test, et_test_pred)
et_test_rmse = np.sqrt(mean_squared_error(y_test, et_test_pred))

print("Final Test Results — Extra Trees")
print("Test R2:  ", round(et_test_r2, 4))
print("Test RMSE:", round(et_test_rmse, 4))

Final Test Results — Extra Trees
Test R2:   0.8476
Test RMSE: 14.3679


# Step 14: Targeted Experiment — Fiber vs No Fiber

- We investigate if the model performs differently on mixes with and without fiber
- Fiber affects concrete strength significantly
- We compare RMSE for fiber and no-fiber groups

In [128]:
# Targeted Experiment: Fiber vs No Fiber

# Create test dataframe with predictions
test_df = X_test.copy()
test_df["actual"] = y_test.values
test_df["predicted"] = et_test_pred

# Split into fiber and no-fiber groups
fiber_mask = test_df["fiber1_amount"] > 0

fiber_group = test_df[fiber_mask]
no_fiber_group = test_df[~fiber_mask]

# Calculate RMSE for each group
fiber_rmse = np.sqrt(mean_squared_error(
    fiber_group["actual"], fiber_group["predicted"]
))
no_fiber_rmse = np.sqrt(mean_squared_error(
    no_fiber_group["actual"], no_fiber_group["predicted"]
))

print("Fiber mixes    — RMSE:", round(fiber_rmse, 4))
print("No Fiber mixes — RMSE:", round(no_fiber_rmse, 4))
print()
print("Fiber mixes count:   ", len(fiber_group))
print("No Fiber mixes count:", len(no_fiber_group))

Fiber mixes    — RMSE: 14.7283
No Fiber mixes — RMSE: 13.6322

Fiber mixes count:    275
No Fiber mixes count: 140


# Step 15: Targeted Experiment — High vs Low Strength

- We investigate if the model performs differently on high and low strength mixes
- High strength: above 150 MPa
- Low strength: below 150 MPa
- We compare RMSE for both groups

In [129]:
# Targeted Experiment: High vs Low Strength

# Split test set into high and low strength groups
high_mask = test_df["actual"] >= 150

high_group = test_df[high_mask]
low_group = test_df[~high_mask]

# Calculate RMSE for each group
high_rmse = np.sqrt(mean_squared_error(
    high_group["actual"], high_group["predicted"]
))
low_rmse = np.sqrt(mean_squared_error(
    low_group["actual"], low_group["predicted"]
))

print("High Strength (>=150 MPa) — RMSE:", round(high_rmse, 4))
print("Low Strength  (<150 MPa)  — RMSE:", round(low_rmse, 4))
print()
print("High strength count:", len(high_group))
print("Low strength count: ", len(low_group))

High Strength (>=150 MPa) — RMSE: 15.7322
Low Strength  (<150 MPa)  — RMSE: 13.0927

High strength count: 191
Low strength count:  224


# Step 16: Final Summary

- Decision Tree baseline: R2 = 0.70, RMSE = 20.56 MPa
- Random Forest: R2 = 0.84, RMSE = 14.97 MPa
- Extra Trees: R2 = 0.86, RMSE = 13.92 MPa (best model)
- Final test score: R2 = 0.85, RMSE = 14.38 MPa
- Model performs better on no-fiber and low strength mixes
- Better encoding reduced prediction gaps significantly

In [130]:
# Final Summary of all results

print("WEEK 7 — FINAL RESULTS SUMMARY (Shared Dataset)")
print()
print("Model Comparison on Validation Data:")
print(f"  Decision Tree  R2: {dt_r2:.4f}   RMSE: {dt_rmse:.2f} MPa")
print(f"  Random Forest  R2: {rf_r2:.4f}   RMSE: {rf_rmse:.2f} MPa")
print(f"  Extra Trees    R2: {et_r2:.4f}   RMSE: {et_rmse:.2f} MPa")
print()
print("Best Model — Extra Trees on Test Set:")
print(f"  Test R2:   {et_test_r2:.4f}")
print(f"  Test RMSE: {et_test_rmse:.2f} MPa")
print()
print("Targeted Experiments:")
print(f"  Fiber mixes     RMSE: {fiber_rmse:.2f} MPa")
print(f"  No Fiber mixes  RMSE: {no_fiber_rmse:.2f} MPa")
print(f"  High Strength   RMSE: {high_rmse:.2f} MPa")
print(f"  Low Strength    RMSE: {low_rmse:.2f} MPa")
print()
print("Key Findings:")
print("  1. Extra Trees outperforms Random Forest on shared dataset")
print("  2. Proper encoding reduced fiber prediction gap from 2.44 to 1.05 MPa")
print("  3. High strength mixes still harder to predict")
print("  4. Shared dataset with encoding gives R2 of 0.85 vs 0.70 on own data")

WEEK 7 — FINAL RESULTS SUMMARY (Shared Dataset)

Model Comparison on Validation Data:
  Decision Tree  R2: 0.7208   RMSE: 19.94 MPa
  Random Forest  R2: 0.8398   RMSE: 15.11 MPa
  Extra Trees    R2: 0.8624   RMSE: 14.00 MPa

Best Model — Extra Trees on Test Set:
  Test R2:   0.8476
  Test RMSE: 14.37 MPa

Targeted Experiments:
  Fiber mixes     RMSE: 14.73 MPa
  No Fiber mixes  RMSE: 13.63 MPa
  High Strength   RMSE: 15.73 MPa
  Low Strength    RMSE: 13.09 MPa

Key Findings:
  1. Extra Trees outperforms Random Forest on shared dataset
  2. Proper encoding reduced fiber prediction gap from 2.44 to 1.05 MPa
  3. High strength mixes still harder to predict
  4. Shared dataset with encoding gives R2 of 0.85 vs 0.70 on own data


# Step 17: Comparison — Own Dataset vs Shared Dataset

In [131]:
# Comparison between own dataset and shared dataset results

print("COMPARISON — OWN DATASET vs SHARED DATASET")
print()
print("Dataset Info:")
print("  Own Data    : 23 features, text dropped")
print("  Shared Data : 116 features, text encoded")
print()
print("Model Results:")
print(f"{'Model':<20} {'Own R2':>8} {'Own RMSE':>10} {'Shared R2':>10} {'Shared RMSE':>12}")
print("-" * 65)
print(f"{'Decision Tree':<20} {'0.5002':>8} {'26.68 MPa':>10} {dt_r2:.4f}  {dt_rmse:.2f} MPa")
print(f"{'Random Forest':<20} {'0.6601':>8} {'22.00 MPa':>10} {rf_r2:.4f}  {rf_rmse:.2f} MPa")
print(f"{'Extra Trees':<20} {'0.6103':>8} {'23.56 MPa':>10} {et_r2:.4f}  {et_rmse:.2f} MPa")
print()
print("Best Model Test Results:")
print(f"  Own Data    : Random Forest  R2: 0.7036  RMSE: 20.04 MPa")
print(f"  Shared Data : Extra Trees    R2: {et_test_r2:.4f}  RMSE: {et_test_rmse:.2f} MPa")
print()
print("Targeted Experiments:")
print(f"{'Experiment':<25} {'Own RMSE':>10} {'Shared RMSE':>12} {'Improvement':>12}")
print("-" * 65)
print(f"{'Fiber mixes':<25} {'20.83 MPa':>10} {fiber_rmse:.2f} MPa  {20.83 - fiber_rmse:.2f} MPa better")
print(f"{'No Fiber mixes':<25} {'18.39 MPa':>10} {no_fiber_rmse:.2f} MPa  {18.39 - no_fiber_rmse:.2f} MPa better")
print(f"{'High Strength':<25} {'23.35 MPa':>10} {high_rmse:.2f} MPa  {23.35 - high_rmse:.2f} MPa better")
print(f"{'Low Strength':<25} {'16.70 MPa':>10} {low_rmse:.2f} MPa  {16.70 - low_rmse:.2f} MPa better")
print()
print("Key Finding:")
print("  Proper encoding improved R2 from 0.70 to 0.85")
print("  Extra Trees benefits more from higher feature dimensions")
print("  Encoding fiber type reduced fiber prediction gap significantly")

COMPARISON — OWN DATASET vs SHARED DATASET

Dataset Info:
  Own Data    : 23 features, text dropped
  Shared Data : 116 features, text encoded

Model Results:
Model                  Own R2   Own RMSE  Shared R2  Shared RMSE
-----------------------------------------------------------------
Decision Tree          0.5002  26.68 MPa 0.7208  19.94 MPa
Random Forest          0.6601  22.00 MPa 0.8398  15.11 MPa
Extra Trees            0.6103  23.56 MPa 0.8624  14.00 MPa

Best Model Test Results:
  Own Data    : Random Forest  R2: 0.7036  RMSE: 20.04 MPa
  Shared Data : Extra Trees    R2: 0.8476  RMSE: 14.37 MPa

Targeted Experiments:
Experiment                  Own RMSE  Shared RMSE  Improvement
-----------------------------------------------------------------
Fiber mixes                20.83 MPa 14.73 MPa  6.10 MPa better
No Fiber mixes             18.39 MPa 13.63 MPa  4.76 MPa better
High Strength              23.35 MPa 15.73 MPa  7.62 MPa better
Low Strength               16.70 MPa 13.09 MP